# 🔬 Universal Federated MARL Run Analysis Notebook
**Framework**: Federated Active Causal Discovery with IPPO  
**Instructions**: Simply set your WandB `RUN_PATH` in Section 1 below and run all cells sequentially to generate exhaustive empirical diagnostic insights.

---

## Section 1: Notebook Initialization & Run Selection

In [ ]:
import wandb
import json
import os
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Markdown

# Parameterizable WandB Run Path
RUN_PATH = "bezinwoke-university-college-london/federated-causal-marl-kaggle/05n0rr9e"
OUTPUT_DIR = "scratch"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Target WandB Run initialized: {RUN_PATH}")


## Section 2: Automated WandB Data Extraction

In [ ]:
api = wandb.Api()
try:
    run = api.run(RUN_PATH)
    config = run.config
    summary = dict(run.summary)
    history_records = list(run.scan_history())
    
    try:
        file = run.file("evaluation_trace.json")
        file.download(root=OUTPUT_DIR, replace=True)
        with open(os.path.join(OUTPUT_DIR, "evaluation_trace.json"), "r") as f:
            trace = json.load(f)
    except Exception as e:
        print(f"Warning: Could not fetch evaluation_trace.json: {e}")
        trace = {}
        
    print(f"Successfully extracted data! Total history steps logged: {len(history_records)}")
except Exception as e:
    print(f"Error connecting to WandB: {e}")


## Section 3: Hyperparameter & System Setup Table

In [ ]:
hyperparam_table = "| Parameter | Value |\n|---|---|\n"
for k, v in sorted(config.items()):
    hyperparam_table += f"| **{k}** | `{v}` |\n"

display(Markdown("### ⚙️ Run Hyperparameters & Environment Setup\n" + hyperparam_table))


## Section 4: Executive Metric Summary

In [ ]:
summary_table = "| Metric | Final / Best Value |\n|---|---|\n"
for k, v in summary.items():
    if not k.startswith("_"):
        val_str = f"{v:.4f}" if isinstance(v, float) else f"{v}"
        summary_table += f"| **{k}** | `{val_str}` |\n"

display(Markdown("### 📊 Executive Summary Metrics\n" + summary_table))


## Section 5: Complete Tabular Metric Progression

In [ ]:
history_table = "| Episode | Reward | Actor Loss | Critic Loss | Graph BCE Loss | Policy Entropy | Eval SHD | Eval F1 | A0 Budget | A1 Budget |\n"
history_table += "|---|---|---|---|---|---|---|---|---|---|\n"

for h in history_records:
    ep = h.get("train/episode")
    if ep and (ep % 10 == 0 or ep == 1 or ep == len(history_records)):
        al = f"{h.get('train/actor_loss', 0.0):.4f}" if h.get('train/actor_loss') is not None else "N/A"
        cl = f"{h.get('train/critic_loss', 0.0):.4f}" if h.get('train/critic_loss') is not None else "N/A"
        gl = f"{h.get('train/graph_loss', 0.0):.4f}" if h.get('train/graph_loss') is not None else "N/A"
        ent = f"{h.get('train/entropy', 0.0):.4f}" if h.get('train/entropy') is not None else "N/A"
        rew = f"{h.get('train/episode_reward', 0.0):.1f}" if h.get('train/episode_reward') is not None else "N/A"
        shd = f"{h.get('eval/shd', 0.0):.1f}" if h.get('eval/shd') is not None else "N/A"
        f1 = f"{h.get('eval/f1', 0.0):.4f}" if h.get('eval/f1') is not None else "N/A"
        b0 = f"{h.get('agent_0_budget', 0.0):.1f}" if h.get('agent_0_budget') is not None else "N/A"
        b1 = f"{h.get('agent_1_budget', 0.0):.1f}" if h.get('agent_1_budget') is not None else "N/A"
        history_table += f"| **{ep}** | **{rew}** | {al} | {cl} | {gl} | {ent} | **{shd}** | **{f1}** | {b0} | {b1} |\n"

display(Markdown("### 📈 Training Progression Table (Sampled Every 10 Episodes)\n" + history_table))


## Section 6: Reward Progression & Convergence Curve

In [ ]:
episodes = [h["train/episode"] for h in history_records if "train/episode" in h]
rewards = [h["train/episode_reward"] for h in history_records if "train/episode_reward" in h]

plt.figure(figsize=(10, 5))
plt.plot(episodes, rewards, color='crimson', linewidth=2, label='Episode Reward')
if len(rewards) >= 5:
    window = min(10, len(rewards))
    rolling_mean = np.convolve(rewards, np.ones(window)/window, mode='valid')
    plt.plot(episodes[window-1:], rolling_mean, color='black', linestyle='--', linewidth=2, label=f'{window}-Ep Rolling Avg')

plt.title('Section 6: Reward Progression & Convergence Curve', fontsize=14, fontweight='bold')
plt.xlabel('Episode', fontsize=12)
plt.ylabel('Episode Reward', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Section 7: Multi-Head Loss Component Trajectories

In [ ]:
actor_loss = [h["train/actor_loss"] for h in history_records if "train/actor_loss" in h]
critic_loss = [h["train/critic_loss"] for h in history_records if "train/critic_loss" in h]
graph_loss = [h["train/graph_loss"] for h in history_records if "train/graph_loss" in h]

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].plot(episodes[:len(actor_loss)], actor_loss, color='blue', linewidth=2)
axs[0].set_title('Actor Surrogate Loss')
axs[0].set_xlabel('Episode')

axs[1].plot(episodes[:len(critic_loss)], critic_loss, color='orange', linewidth=2)
axs[1].set_title('Critic Value Loss')
axs[1].set_xlabel('Episode')

axs[2].plot(episodes[:len(graph_loss)], graph_loss, color='green', linewidth=2)
axs[2].set_title('Graph Head BCE Loss')
axs[2].set_xlabel('Episode')

for ax in axs:
    ax.grid(True, alpha=0.3)

plt.suptitle('Section 7: Multi-Head Loss Component Trajectories', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 8: Policy Exploration & Entropy Decay

In [ ]:
entropies = [h["train/entropy"] for h in history_records if "train/entropy" in h]

plt.figure(figsize=(10, 5))
plt.plot(episodes[:len(entropies)], entropies, color='purple', linewidth=2)
plt.title('Section 8: Policy Entropy Trajectory', fontsize=14, fontweight='bold')
plt.xlabel('Episode', fontsize=12)
plt.ylabel('Policy Entropy', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Section 9: Multi-Agent Budget Consumption Profile

In [ ]:
a0_b = [h["agent_0_budget"] for h in history_records if "agent_0_budget" in h]
a1_b = [h["agent_1_budget"] for h in history_records if "agent_1_budget" in h]

plt.figure(figsize=(10, 5))
if a0_b:
    plt.plot(episodes[:len(a0_b)], a0_b, label='Agent 0 Rem Budget', color='darkcyan', linewidth=2)
if a1_b:
    plt.plot(episodes[:len(a1_b)], a1_b, label='Agent 1 Rem Budget', color='magenta', linestyle='--', linewidth=2)

plt.title('Section 9: Multi-Agent Remaining Budget Profile', fontsize=14, fontweight='bold')
plt.xlabel('Episode', fontsize=12)
plt.ylabel('Remaining Budget (Initial = 20.0)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Section 10: Structural Discovery Performance Trajectories

In [ ]:
eval_shds = [h["eval/shd"] for h in history_records if "eval/shd" in h]
eval_f1s = [h["eval/f1"] for h in history_records if "eval/f1" in h]

fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('Eval SHD (Lower is better)', color=color, fontsize=12)
ax1.plot(episodes[:len(eval_shds)], eval_shds, color=color, linewidth=2, marker='o')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Eval F1 Score (Higher is better)', color=color, fontsize=12)
ax2.plot(episodes[:len(eval_f1s)], eval_f1s, color=color, linewidth=2, linestyle='--', marker='s')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Section 10: Structural Discovery Checkpoint Metrics (SHD & F1)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Section 11: 8-Topology Evaluation Matrix

In [ ]:
topo_names = [
    "Chain (Z1 -> X1 -> X2 -> Z2)",
    "Reversed Chain (Z1 <- X1 <- X2 <- Z2)",
    "Collider (Z1 -> X1 <- X2 <- Z2)",
    "Reversed Collider (Z1 -> X1 -> X2 <- Z2)",
    "Fork (Z1 <- X1 -> X2 -> Z2)",
    "Reversed Fork (Z1 <- X1 <- X2 -> Z2)",
    "Fork + Collider (Z1 -> X1 <- X2 -> Z2)",
    "Reversed Fork + Collider (Z1 <- X1 -> X2 <- Z2)"
]

eval_matrix_table = "| Graph ID | Topology Name | Final SHD | Discovery Status | Agent 0 Action Pattern (Steps 1-5) | Agent 1 Action Pattern (Steps 1-5) |\n"
eval_matrix_table += "|---|---|---|---|---|---|\n"

if trace:
    for idx in range(8):
        key = f"graph_{idx}"
        if key in trace:
            data = trace[key]
            final_shd = data["steps"][-1]["shd"]
            status = "🏆 PERFECT DISCOVERY" if final_shd == 0.0 else ("⚠️ PARTIAL" if final_shd < 4.0 else "❌ FAILED")
            a0_p = ", ".join([f"C{s['actions']['agent_0']['cat']}T{s['actions']['agent_0']['target']}" for s in data["steps"][:5]])
            a1_p = ", ".join([f"C{s['actions']['agent_1']['cat']}T{s['actions']['agent_1']['target']}" for s in data["steps"][:5]])
            eval_matrix_table += f"| **{idx}** | {topo_names[idx]} | **{final_shd:.1f}** | {status} | `{a0_p}` | `{a1_p}` |\n"
else:
    eval_matrix_table += "| N/A | Trace file not found | N/A | N/A | N/A | N/A |\n"

display(Markdown("### 🗺️ Section 11: Post-Training 8-Topology Evaluation Matrix\n" + eval_matrix_table))


## Section 12: Step-by-Step Per-Topology SHD Progression

In [ ]:
if trace:
    plt.figure(figsize=(12, 6))
    for graph_key, data in trace.items():
        steps = [s["step"] for s in data["steps"]]
        shd_vals = [s["shd"] for s in data["steps"]]
        plt.plot(steps, shd_vals, marker='o', linewidth=2, label=f"{graph_key}: Final SHD {shd_vals[-1]}")
    
    plt.title('Section 12: Step-by-Step SHD Reduction Across All 8 Topologies', fontsize=14, fontweight='bold')
    plt.xlabel('Environment Step (0-20)', fontsize=12)
    plt.ylabel('Structural Hamming Distance (SHD)', fontsize=12)
    plt.xticks(range(0, 21))
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Trace data not available for step-by-step SHD progression plot.")


## Section 13: Agent Action & Target Distribution Audit

In [ ]:
if trace:
    cat_counts = {"agent_0": {0: 0, 1: 0, 2: 0}, "agent_1": {0: 0, 1: 0, 2: 0}}
    target_counts = {"agent_0": {0: 0, 1: 0, 2: 0, 3: 0}, "agent_1": {0: 0, 1: 0, 2: 0, 3: 0}}
    
    for graph_key, data in trace.items():
        for step in data["steps"]:
            for ag in ["agent_0", "agent_1"]:
                c = step["actions"][ag]["cat"]
                t = step["actions"][ag]["target"]
                cat_counts[ag][c] += 1
                target_counts[ag][t] += 1
                
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    cats = ['0: Local', '1: Peer', '2: NOOP']
    x = np.arange(len(cats))
    width = 0.35
    
    axs[0].bar(x - width/2, [cat_counts['agent_0'][i] for i in range(3)], width, label='Agent 0', color='royalblue')
    axs[0].bar(x + width/2, [cat_counts['agent_1'][i] for i in range(3)], width, label='Agent 1', color='darkorange')
    axs[0].set_title('Action Category Selection Distribution')
    axs[0].set_xticks(x)
    axs[0].set_xticklabels(cats)
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)
    
    targets = ['Node 0 (Z1)', 'Node 1 (X1)', 'Node 2 (X2)', 'Node 3 (Z2)']
    xt = np.arange(len(targets))
    axs[1].bar(xt - width/2, [target_counts['agent_0'][i] for i in range(4)], width, label='Agent 0', color='royalblue')
    axs[1].bar(xt + width/2, [target_counts['agent_1'][i] for i in range(4)], width, label='Agent 1', color='darkorange')
    axs[1].set_title('Target Node Selection Distribution')
    axs[1].set_xticks(xt)
    axs[1].set_xticklabels(targets, rotation=15)
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)
    
    plt.suptitle('Section 13: Agent Action & Target Selection Breakdown', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Trace data not available for action distribution analysis.")


## Section 14: Causal DAG & Markov Equivalence Diagnostic

In [ ]:
if trace:
    colliders_shd = []
    chains_forks_shd = []
    
    for idx in range(8):
        key = f"graph_{idx}"
        if key in trace:
            shd = trace[key]["steps"][-1]["shd"]
            if idx in [2, 3, 6]:
                colliders_shd.append(shd)
            else:
                chains_forks_shd.append(shd)
                
    avg_colliders = np.mean(colliders_shd) if colliders_shd else 0.0
    avg_chains_forks = np.mean(chains_forks_shd) if chains_forks_shd else 0.0
    
    diag_text = f"""### 🧩 Section 14: Markov Equivalence Diagnostic Output
- **Mean SHD on Observational Colliders (Graphs 2, 3, 6)**: `{avg_colliders:.2f}`
- **Mean SHD on Interventional Chains/Forks (Graphs 0, 1, 4, 5, 7)**: `{avg_chains_forks:.2f}`

#### Interpretation:
"""
    if avg_colliders < avg_chains_forks:
        diag_text += "✅ **Observational Dominance**: The agent successfully exploits observational covariance signatures (unshielded colliders), but struggles to execute active interventions to break Markov Equivalence for Chains/Forks."
    else:
        diag_text += "⚖️ **Balanced Discovery**: The agent shows uniform discovery performance across both observational colliders and interventional chains/forks."
        
    display(Markdown(diag_text))


## Section 15: Action Target Validity & Masking Audit

In [ ]:
if trace:
    invalid_attempts = {"agent_0": 0, "agent_1": 0}
    for graph_key, data in trace.items():
        for step in data["steps"]:
            a0 = step["actions"]["agent_0"]
            a1 = step["actions"]["agent_1"]
            if a0["cat"] == 0 and a0["target"] not in [0, 1]:
                invalid_attempts["agent_0"] += 1
            if a1["cat"] == 0 and a1["target"] not in [2, 3]:
                invalid_attempts["agent_1"] += 1
                
    audit_text = f"""### 🛡️ Section 15: Action Target Masking Audit Results
- **Agent 0 Invalid Local Target Attempts**: `{invalid_attempts['agent_0']}`
- **Agent 1 Invalid Local Target Attempts**: `{invalid_attempts['agent_1']}`

#### Audit Assessment:
"""
    if invalid_attempts["agent_0"] > 0 or invalid_attempts["agent_1"] > 0:
        audit_text += "🚨 **Masking Defect Detected**: The policy attempted local interventions on unowned variables! Ensure `mask_invalid_targets` is using strict local ownership masks (`[1, 1, 0, 0]` and `[0, 0, 1, 1]`)."
    else:
        audit_text += "✅ **Strict Masking Verified**: All local intervention actions targeted valid, owned nodes."
        
    display(Markdown(audit_text))
